## Messaging

This example demonstrates the Messenger functionality in MMM-Audio, and the many kinds of messages that can be sent from Python to Mojo to control parameters in the audio graph.

We are able to send:
- Boolean values - .send_bool()
- Float values - .send_float()
- Lists of floats - .send_floats()
- Integer values - .send_int()
- Lists of integers - .send_ints()
- String values - .send_string() 
- Lists of strings - .send_strings()
- Trigger messages - .send_trig()

In [ ]:
import os
# Move up one level to the parent directory
os.chdir('..') 
print(os.getcwd())

In [ ]:
from mmm_python import *

mmm_audio_instance = MMMAudio(128, graph_name="MessengerExample", package_name="examples")

mmm_audio_instance.start_audio()

## In Mojo

To use the ```Messenger``` struct in Mojo, an instance of it is stored in a struct property of the target graph. 

```mojo

struct MessengerExample(Copyable, Movable):
    var world: World
    var m: Messenger # <---- The property to store the Messenger instance in
    var bool: Bool

    def __init__(out self, world: World):
        self.world = world
        self.m = Messenger(self.world) # The Messenger instance being created
        self.bool = False
```

The messages can then be handled in the the graph's ```next()``` method using the following ```Messenger``` methods:

* .update(address, variable) - Updates the value stored in the given variable when a new message is received on the given address.
* .notify_update(address, variable) - Updates the value stored in the given variable when a new message is received on the given address and returns True.
* .notify_trig(address) - Returns True when a new trigger message is received on the address.

```mojo
def next(mut self) -> MFloat[2]:

        # This updates the value stored in the self.bool property and then returns True, causing the if statement to execute.
        if self.m.notify_update("bool", self.bool) :
            print("Bool value is now: " + String(self.bool))

        # This returns True, causing the if statement to execute.
        if self.m.notify_trig("trig"):
            print("Received trig")
```

New messages are processed at the beginning of every block, meaning that larger block sizes introduce some latency in the receipt of messages. Additionally, this causes only the last received message on an address each block to be to be executed. Any previous messages will be ignored:


In [ ]:
mmm_audio_instance.send_bool("bool", True)
mmm_audio_instance.send_bool("bool", False) #Only this message is used

In [ ]:
mmm_audio_instance.send_bool("bool", False)
mmm_audio_instance.send_bool("bool", True) #Only this message is used

There are a few approaches to address this. If you need to control multiple synths using the same address ("freq" for example), you can target them specifically by specifying a *namespace* when creating a ```Messenger```. This can be seen in the Tone struct in the MessengerExamples.mojo file:

```mojo
struct Tone(Movable,Copyable):
    var world: World
    var m: Messenger
    var freq: Float64
    var gate: Bool

    def __init__(out self, world: World, namespace: String):
        self.world = world
        self.m = Messenger(self.world,namespace)
        self.freq = 440.0
        self.gate = False

    def next(mut self) -> Float64:

        if self.m.notify_update("freq", self.freq) :
            print("Tone freq updated to ", self.freq)

        if self.m.notify_update("gate", self.gate) :
            print("Tone gate updated to ", self.gate)
```

```mojo
struct MessengerExample(Copyable, Movable):
    var world: World
    var m: Messenger
    var tones: List[Tone]

    def __init__(out self, world: World):
        self.world = world
        self.m = Messenger(self.world)

        self.tones = List[Tone]()
        # The for loop below assigns a unique namespace (tone_0, and tone_1) for each instance of Tone
        for i in range(2):
            self.tones.append(Tone(self.world, "tone_" + String(i)))

```

The namespace can then be targetted by the .send_x() methods by separating the namespace and address with a period (.)

In [ ]:

# Starts the synths
mmm_audio_instance.send_bool("tone_0.gate",True)
mmm_audio_instance.send_bool("tone_1.gate",True)

In [ ]:
# Sends a message to each Tone's "freq" address
mmm_audio_instance.send_float("tone_0.freq",440 * 1.059)
mmm_audio_instance.send_float("tone_1.freq",midicps(74))

In [ ]:
# Stops the synths
mmm_audio_instance.send_bool("tone_0.gate",False)
mmm_audio_instance.send_bool("tone_1.gate",False)

## In Python

MMMAudio's ```Messenger``` struct allows for messages to be sent from Python to Mojo. This allows for the control of the compiled Mojo code with interpreted Python code. 

It's possible to send boolean, integer, floats, strings, and triggers as well as lists of integers, floats, or strings. This is done by calling one of the following methods on an MMMAudio instance:

Single pieces of data
* .send_bool(adress, value)
* .send_int(address, value)
* .send_float(address, value)
* .send_string(address, value)
* .send_trig(address)

Lists of data
* .send_ints(address, [values])
* .send_floats(address, [values])
* .send_strings(address, [values])

Try sending some messages to Mojo below:

In [ ]:
mmm_audio_instance.send_bool("bool",True)  

In [ ]:
mmm_audio_instance.send_float("float", 440.0)

In [ ]:
mmm_audio_instance.send_floats("floats", [440.0, 550.0, 660.0])

In [ ]:
mmm_audio_instance.send_int("int", 42)

In [ ]:
mmm_audio_instance.send_ints("ints", [1, 22, 3, 4, 5])

In [ ]:
mmm_audio_instance.send_string("string", "Hello, World!")

In [ ]:
mmm_audio_instance.send_strings("strings", ["hello", "there", "general", "kenobi"])

In [ ]:
mmm_audio_instance.send_trig("trig")